# Step 1.1: Aggregating Patient Cancer Registration Records

## Overview
Consolidates distributed cancer registration EMR files. It scans patient data folders, extracts columns containing diagnostics, staging classifications, and patient age, and merges them into a single cohort dataset.

## Inputs and Outputs
- **Input**: EMR directories matching: `../../data/step1_dwh_outputs/p1_pts_parse_whole_spilt_2025_/d*`
- **Output**: `../../data/step1_dwh_outputs/step1_all_cancer.parquet`



In [ ]:
import pandas as pd
import glob
import os
import re

# ==========================================
# 1. Paths and Configuration
# ==========================================
data_dir = "../../data" 
base_input_dir = f"{data_dir}/step1_dwh_outputs/p1_pts_parse_whole_spilt_2025_"
output_dir = f"{data_dir}/step1_dwh_outputs"

# List to store DataFrames
df_list = []

# ==========================================
# 2. Folder Traversal and Loop Processing
# ==========================================
# Get target directories (d00, d01, ...)
target_dirs = sorted(glob.glob(os.path.join(base_input_dir, "d*")))

print(f"Detected target directories: {len(target_dirs)}")

for d_path in target_dirs:
    # Extract batch ID (e.g., "d00") from directory path
    batch_id = os.path.basename(d_path)
    
    # Check if batch ID matches "d" + digits pattern
    if not re.match(r'^d\d+$', batch_id):
        continue

    # Generate file path pattern
    # Search for files replacing "d00" with batch_id
    search_pattern = os.path.join(d_path, f"df_{batch_id}_tCANCER_REGISTRATION_*.parquet")
    found_files = glob.glob(search_pattern)

    if not found_files:
        print(f"Warning: {batch_id} folder contains no target files. Skipping.")
        continue
    
    # Use the first file if multiple files match the pattern
    file_path = found_files[0]

    try:
        # Read Parquet file
        df = pd.read_parquet(file_path)

        # Select target columns
        df_canc = df.loc[
            :,
            [
                "hs",
                "PT_ID",
                "DIAGNOSIS_CD",
                "DIAGNOSIS_TEXT",
                "UICC_STAGE_BEFORE",
                "MAIN5_STAGE_BEFORE",
                "UICC_STAGE_AFTER",
                "ICD_O_3_CD",
                "ICD_O_3_TEXT",
                "DIAGNOSIS_DATE_1",
                "DIAGNOSIS_DATE_2",
                "AGE",
            ]
        ]


        # Append to the list
        df_list.append(df_canc)

    except Exception as e:
        print(f"Error: {batch_id} encountered an issue during processing: {e}")

# ==========================================
# 3. Data Aggregation and Output
# ==========================================
if df_list:
    # Vertically concatenate all DataFrames
    df_all = pd.concat(df_list, ignore_index=True)
    
    # Create output directory if it does not exist
    os.makedirs(output_dir, exist_ok=True)

    # Export as Parquet file
    output_filename = "step1_all_cancer.parquet"
    output_path = os.path.join(output_dir, output_filename)
    
    df_all.to_parquet(output_path, index=False)
    
    print("-" * 30)
    print(f"Processing complete: All data combined.")
    print(f"Output file: {output_path}")
    print(f"Total rows: {len(df_all)}")
else:
    print("No target data was found to process.")